In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv("./archive/UrbanSound8K.csv")
df.head()

,slice_file_name,fsID,start,end,salience,fold,classID,class
0,100032-3-0-0.wav,100032,0.0,0.317551,1,5,3,dog_bark
1,100263-2-0-117.wav,100263,58.5,62.500000,1,5,2,children_playing
2,100263-2-0-121.wav,100263,60.5,64.500000,1,5,2,children_playing
3,100263-2-0-126.wav,100263,63.0,67.000000,1,5,2,children_playing
4,100263-2-0-137.wav,100263,68.5,72.500000,1,5,2,children_playing


In [3]:
import os
import librosa
import librosa.display

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

In [5]:
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, Dropout

In [6]:
# Audio Playback (Optional)
import IPython.display as ipd

# Resampling Library
import resampy  

from tqdm.auto import tqdm

c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
ext_df = pd.DataFrame(extracted,columns=['feature','class'])
ext_df

,feature,class
0,"[-646.75635, 5.3873677, -9.81038, -4.852548, -...",dog_bark
1,"[-490.71515, 98.969734, -42.70029, 51.26325, 9...",children_playing
2,"[-525.64197, 111.47854, -37.607727, 43.46369, ...",children_playing
3,"[-480.8334, 91.745125, -24.20152, 42.91354, 11...",children_playing
4,"[-512.5282, 103.20618, -42.739483, 50.844624, ...",children_playing
...,...,...
8727,"[-463.6058, 124.70258, -40.755966, 25.904419, ...",car_horn
8728,"[-632.8646, 41.30297, -18.6657, 22.987225, -12...",car_horn
8729,"[-429.492, 90.46378, -32.355354, 23.182915, -0...",car_horn
8730,"[-597.676, 73.3897, -28.41001, 15.767284, -6.7...",car_horn


In [12]:
x = np.array(ext_df['feature'].tolist())
y = np.array(ext_df['class'].tolist())
le = LabelEncoder()

y = to_categorical(le.fit_transform(y))
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state = 42)

print("Number of training samples = ", x_train.shape[0])
print("Number of testing samples = ",x_test.shape[0])

Number of training samples =  6985
Number of testing samples =  1747


In [ ]:
import tensorflow as tf
import os
import numpy as np
import librosa
import librosa.display
import pandas as pd
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

def extract_features(file_path, max_pad_len=174, max_duration=4):
    try:
        audio, sample_rate = librosa.load(file_path, sr=None, res_type='kaiser_fast', duration=max_duration)
        mel_spec = librosa.feature.melspectrogram(y=audio, sr=sample_rate, n_mels=128)
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
        pad_width = max_pad_len - mel_spec_db.shape[1]
        if pad_width > 0:
            mel_spec_db = np.pad(mel_spec_db, ((0, 0), (0, pad_width)), mode='constant')
        else:
            mel_spec_db = mel_spec_db[:, :max_pad_len]
        return mel_spec_db
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

def load_data(dataset_path):
    metadata = pd.read_csv(os.path.join(dataset_path, 'UrbanSound8K.csv'))
    features, labels = [], []
    for index, row in metadata.iterrows():
        file_path = os.path.join(dataset_path, 'fold' + str(row["fold"]), row["slice_file_name"])
        feature = extract_features(file_path)
        if feature is not None:
            features.append(feature)
            labels.append(row["class"])
    return np.array(features), np.array(labels)

dataset_path = "./archive"
X, y = load_data(dataset_path)
le = LabelEncoder()
y = le.fit_transform(y)
X = np.transpose(X, (0, 2, 1))  # Swap dimensions to match (time, frequency)
X = X[..., np.newaxis]  

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)




c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\librosa\feature\spectral.py:2143: UserWarning: Empty filters detected in mel frequency basis. Some channels will produce empty responses. Try increasing your sampling rate (and fmax) or reducing n_mels.
  mel_basis = filters.mel(sr=sr, n_fft=n_fft, **kwargs)


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, Input
def create_resnet_se(num_class, input_size, num_blocks=[3, 4, 6, 3], num_filters=[32, 64, 128, 256], embd_dim=192):
    inputs = Input(shape=(input_size, 128, 1))
    x = layers.Conv2D(num_filters[0], (3, 3), strides=(1, 1), padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    expansion = 2

    def res_block(x, filters, blocks, stride=1):
        for i in range(blocks):
            shortcut = x
            x = layers.Conv2D(filters, (1, 1), use_bias=False, strides=(stride if i == 0 else 1))(x)
            x = layers.BatchNormalization()(x)
            x = layers.ReLU()(x)
            x = layers.Conv2D(filters, (3, 3), padding='same', use_bias=False)(x)
            x = layers.BatchNormalization()(x)
            x = layers.ReLU()(x)
            x = layers.Conv2D(filters * expansion, (1, 1), use_bias=False)(x)
            x = layers.BatchNormalization()(x)

            # Squeeze-and-Excitation
            se = layers.GlobalAveragePooling2D()(x)
            se = layers.Dense(filters * expansion // 8, activation='relu')(se)
            se = layers.Dense(filters * expansion, activation='sigmoid')(se)
            se = layers.Reshape((1, 1, filters * expansion))(se)
            x = layers.Multiply()([x, se])

            # Ensure shortcut has the same shape
            if shortcut.shape[-1] != x.shape[-1]:
                shortcut = layers.Conv2D(filters * expansion, (1, 1), strides=stride, use_bias=False)(shortcut)
                shortcut = layers.BatchNormalization()(shortcut)

            x = layers.Add()([shortcut, x])
            x = layers.ReLU()(x)
        return x

    x = res_block(x, num_filters[0], num_blocks[0])
    x = res_block(x, num_filters[1], num_blocks[1], stride=2)
    x = res_block(x, num_filters[2], num_blocks[2], stride=2)
    x = res_block(x, num_filters[3], num_blocks[3], stride=2)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(embd_dim, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    outputs = layers.Dense(num_class, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    return model

model = create_resnet_se(num_class=len(np.unique(y)), input_size=174)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


In [ ]:

history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_data=(X_test, y_test))

Epoch 1/20
  6/219 ━━━━━━━━━━━━━━━━━━━━ 1:46:11 30s/step - accuracy: 0.1728 - loss: 2.6688

In [13]:
def extract_feature(file_name, target_duration=4, sr=22050, n_mfcc=50):
    target_length = target_duration * sr 
    audio_data, sample_rate = librosa.load(file_name, sr=sr, res_type='soxr_vhq') 
    if len(audio_data) < target_length:
        audio_data = np.pad(audio_data, (0, target_length - len(audio_data)), mode='constant')
    fea = librosa.feature.mfcc(y=audio_data, sr=sample_rate, n_mfcc=n_mfcc)
    scaled = np.mean(fea.T, axis=0)

    return np.array([scaled])

def print_prediction(file_name):
    pred_fea = extract_feature(file_name) 
    pred_vector = np.argmax(model.predict(pred_fea), axis=-1)
    pred_class = le.inverse_transform(pred_vector)
    print("The predicted class is:", pred_class[0], '\n') 

In [14]:
df

,slice_file_name,fsID,start,end,salience,fold,classID,class
0,100032-3-0-0.wav,100032,0.000000,0.317551,1,5,3,dog_bark
1,100263-2-0-117.wav,100263,58.500000,62.500000,1,5,2,children_playing
2,100263-2-0-121.wav,100263,60.500000,64.500000,1,5,2,children_playing
3,100263-2-0-126.wav,100263,63.000000,67.000000,1,5,2,children_playing
4,100263-2-0-137.wav,100263,68.500000,72.500000,1,5,2,children_playing
...,...,...,...,...,...,...,...,...
8727,99812-1-2-0.wav,99812,159.522205,163.522205,2,7,1,car_horn
8728,99812-1-3-0.wav,99812,181.142431,183.284976,2,7,1,car_horn
8729,99812-1-4-0.wav,99812,242.691902,246.197885,2,7,1,car_horn
8730,99812-1-5-0.wav,99812,253.209850,255.741948,2,7,1,car_horn


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Conv1D, BatchNormalization, ReLU, Add, GlobalAveragePooling1D, Dense, Dropout, LSTM, MaxPooling1D, Input
from tensorflow.keras.models import Model
import matplotlib.pyplot as plt

# ✅ Define Residual Block
def residual_block(x, filters, kernel_size=3, strides=1):
    shortcut = x  # Save input for residual connection
    x = Conv1D(filters, kernel_size, strides=strides, padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Conv1D(filters, kernel_size, strides=1, padding="same")(x)
    x = BatchNormalization()(x)
    if shortcut.shape[-1] != filters:
        shortcut = Conv1D(filters, 1, strides=strides, padding="same")(shortcut)
    x = Add()([x, shortcut])  # Add skip connection
    x = ReLU()(x)
    return x

# ✅ Build Hybrid ResNet + LSTM Model
def build_hybrid_model(input_shape, num_labels):
    input_layer = Input(shape=input_shape)

    # Initial Conv Layer
    x = Conv1D(64, 3, padding="same", activation='relu')(input_layer)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    
    # Residual Blocks
    x = residual_block(x, 64)
    x = residual_block(x, 128)
    x = residual_block(x, 256)
    
    # CNN Layers (Feature Extraction)
    x = Conv1D(512, kernel_size=3, activation='relu')(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = BatchNormalization()(x)
    
    # LSTM Layers (Temporal Dependencies)
    x = LSTM(256, return_sequences=True)(x)
    x = LSTM(128, return_sequences=False)(x)
    
    # Fully Connected Layers
    x = Dense(512, activation="relu")(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.5)(x)
    x = Dense(128, activation="relu")(x)
    x = Dropout(0.5)(x)
    
    # Output Layer
    output_layer = Dense(num_labels, activation="softmax")(x)
    
    # Define Model
    model = Model(inputs=input_layer, outputs=output_layer)
    return model

# ✅ Define input shape & labels
input_shape = (50, 1)  # Adjust based on MFCC feature shape
num_labels = y_train.shape[1]  # Assuming one-hot encoded labels

# ✅ Create model
model = build_hybrid_model(input_shape, num_labels)
model.summary()

# ✅ Compile model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=['accuracy']
)

# ✅ Train model
history = model.fit(
    x_train, 
    y_train, 
    epochs=200,  
    validation_data=(x_test, y_test),
)

# ✅ Function to Plot Accuracy & Loss
def showAccVallLossPlt(history):
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    ax[0].plot(history.history['accuracy'], label='Train Accuracy')
    ax[0].plot(history.history['val_accuracy'], label='Val Accuracy')
    ax[0].set_title('Model Accuracy')
    ax[0].set_xlabel('Epochs')
    ax[0].set_ylabel('Accuracy')
    ax[0].legend()
    ax[1].plot(history.history['loss'], label='Train Loss')
    ax[1].plot(history.history['val_loss'], label='Val Loss')
    ax[1].set_title('Model Loss')
    ax[1].set_xlabel('Epochs')
    ax[1].set_ylabel('Loss')
    ax[1].legend()
    plt.show()

# ✅ Plot Results
showAccVallLossPlt(history)

In [19]:
history2 = model.fit(
    x_train, 
    y_train, 
    epochs=250,
    initial_epoch = 200,  
    validation_data=(x_test, y_test),
)

model.save_weights('cln..weights.h5')
# model.save_weights('cln.h')
print("Model weights saved successfully in .keras format!")

Epoch 201/250
219/219 ━━━━━━━━━━━━━━━━━━━━ 12s 53ms/step - accuracy: 0.9951 - loss: 0.0203 - val_accuracy: 0.9193 - val_loss: 0.5501
Epoch 202/250
219/219 ━━━━━━━━━━━━━━━━━━━━ 11s 51ms/step - accuracy: 0.9957 - loss: 0.0089 - val_accuracy: 0.9239 - val_loss: 0.5573
Epoch 203/250
219/219 ━━━━━━━━━━━━━━━━━━━━ 11s 51ms/step - accuracy: 0.9964 - loss: 0.0093 - val_accuracy: 0.9193 - val_loss: 0.5386
Epoch 204/250
219/219 ━━━━━━━━━━━━━━━━━━━━ 11s 51ms/step - accuracy: 0.9963 - loss: 0.0090 - val_accuracy: 0.9010 - val_loss: 0.6422
Epoch 205/250
219/219 ━━━━━━━━━━━━━━━━━━━━ 11s 51ms/step - accuracy: 0.9933 - loss: 0.0247 - val_accuracy: 0.9222 - val_loss: 0.5598
Epoch 206/250
219/219 ━━━━━━━━━━━━━━━━━━━━ 11s 52ms/step - accuracy: 0.9976 - loss: 0.0064 - val_accuracy: 0.9244 - val_loss: 0.6091
Epoch 207/250
219/219 ━━━━━━━━━━━━━━━━━━━━ 11s 52ms/step - accuracy: 0.9924 - loss: 0.0311 - val_accuracy: 0.9033 - val_loss: 0.6103
Epoch 208/250
219/219 ━━━━━━━━━━━━━━━━━━━━ 11s 52ms/step - accuracy: 

In [22]:
model.save_weights('cln..weights.h5')
# model.save_weights('cln.h')
print("Model weights saved successfully in .keras format!")

Model weights saved successfully in .keras format!


In [1]:
import tensorflow as tf
from tensorflow.keras.layers import Conv1D, BatchNormalization, ReLU, Add, GlobalAveragePooling1D, Dense, Dropout, LSTM, MaxPooling1D, Input
from tensorflow.keras.models import Model
import matplotlib.pyplot as plt

# ✅ Define Residual Block
def residual_block(x, filters, kernel_size=3, strides=1):
    shortcut = x  # Save input for residual connection
    x = Conv1D(filters, kernel_size, strides=strides, padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Conv1D(filters, kernel_size, strides=1, padding="same")(x)
    x = BatchNormalization()(x)
    if shortcut.shape[-1] != filters:
        shortcut = Conv1D(filters, 1, strides=strides, padding="same")(shortcut)
    x = Add()([x, shortcut])  # Add skip connection
    x = ReLU()(x)
    return x

# ✅ Build Hybrid ResNet + LSTM Model
def build_hybrid_model(input_shape, num_labels):
    input_layer = Input(shape=input_shape)

    # Initial Conv Layer
    x = Conv1D(64, 3, padding="same", activation='relu')(input_layer)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    
    # Residual Blocks
    x = residual_block(x, 64)
    x = residual_block(x, 128)
    x = residual_block(x, 256)
    
    # CNN Layers (Feature Extraction)
    x = Conv1D(512, kernel_size=3, activation='relu')(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = BatchNormalization()(x)
    
    # LSTM Layers (Temporal Dependencies)
    x = LSTM(256, return_sequences=True)(x)
    x = LSTM(128, return_sequences=False)(x)
    
    # Fully Connected Layers
    x = Dense(512, activation="relu")(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.5)(x)
    x = Dense(128, activation="relu")(x)
    x = Dropout(0.5)(x)
    
    # Output Layer
    output_layer = Dense(num_labels, activation="softmax")(x)
    
    # Define Model
    model = Model(inputs=input_layer, outputs=output_layer)
    return model

# ✅ Define input shape & labels
input_shape = (50, 1)  # Adjust based on MFCC feature shape
num_labels = y_train.shape[1]  # Assuming one-hot encoded labels

# ✅ Create model
model = build_hybrid_model(input_shape, num_labels)
model.summary()

NameError: name 'y_train' is not defined

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
class_labels = [
    "Air Conditioner",  # 0
    "Car Horn",         # 1
    "Children Playing", # 2
    "Dog Bark",         # 3
    "Drilling",         # 4
    "Engine Idling",    # 5
    "Gun Shot",         # 6
    "Jackhammer",       # 7
    "Siren",            # 8
    "Street Music"      # 9
]


model.load_weights('cln..weights.h5')
print("Model weights loaded successfully!")

y_pred_probs = model.predict(x_test)  
y_pred = np.argmax(y_pred_probs, axis=1) 

y_true = np.argmax(y_test, axis=1)  # If y_test is one-hot encoded

# Step 3: Compute Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

# Step 4: Plot Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_labels, yticklabels=class_labels)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

# Step 5: Print Classification Report
print("Classification Report:\n", classification_report(y_true, y_pred))


NameError: name 'model' is not defined

In [1]:
# Calculate accuracy
accuracy = accuracy_score(y_true, y_pred)

# Generate classification report
report = classification_report(y_true, y_pred, target_names=class_labels, output_dict=True)

# Extract overall precision, recall, and F1-score
precision = report["weighted avg"]["precision"]
recall = report["weighted avg"]["recall"]
f1_score = report["weighted avg"]["f1-score"]

# Print scores
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1_score:.4f}")


NameError: name 'accuracy_score' is not defined